In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import hnsw_benchmark2 as bm
import farhest_point_order as fp
from datasets import load_dataset
import os

In [ ]:
# Adathalmazok beolvasása
corpus = load_dataset("mteb/arguana", "corpus")
corpus_texts = corpus.data["corpus"]["text"]

queries = load_dataset("mteb/arguana", "queries")
query_texts = queries.data["queries"]["text"]

In [ ]:
print(corpus_texts[0])
# print(query_texts[0])

In [ ]:
# Embedding végrehajtása
model = SentenceTransformer("intfloat/e5-small-v2")

corpus_embeddings = model.encode(
    [f"passage: {text}" for text in corpus_texts],
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True,
)

query_embeddings = model.encode(
    [f"query: {text}" for text in query_texts],
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True,
)


corpus_embeddings = np.asarray(corpus_embeddings, dtype=np.float32)
query_embeddings = np.asarray(query_embeddings, dtype=np.float32)

In [ ]:
print(corpus_embeddings.shape)
print(query_embeddings.shape)

In [ ]:
# LID számolás
corpus_lids = bm.calculate_lid(corpus_embeddings)

In [ ]:
# Farthest points igazi számítása
df_farthest_score = fp.farthest_point_order(corpus_vectors)

# print(df_corpus.keys())
# print(len(df_farthest_score))
# df_farthest_score = np.pad(
#     df_farthest_score,
#     (0, len(df_corpus) - len(df_farthest_score)),
#     mode="constant",
#     constant_values=0,
# )
# print(len(df_farthest_score))
df_corpus = df_corpus.assign(fp_score=df_farthest_score)
# df_corpus.drop(columns=["fp_score"], inplace=True)
print(df_corpus.keys())
df_corpus.to_json(corpus_output_path, orient="records", lines=True)

In [ ]:
df_corpus = pd.DataFrame(columns=["vectors"], data=[[v, lid] for v, lid in zip(corpus_embeddings, corpus_lids)])
df_query = pd.DataFrame(columns=["vectors"], data=[[v] for v in query_embeddings])

dataset_name = "arguana"
corpus_filename = dataset_name + "_corpus_lid"
corpus_output_path = os.path.join("..\\data\\", corpus_filename + ".json")
df_corpus.to_json(corpus_output_path, orient="records", lines=True)

query_filename = dataset_name + "_query"
query_output_path = os.path.join("..\\data\\", query_filename + ".json")
df_query.to_json(query_output_path, orient="records", lines=True)